# 04 — Imágenes satelitales con Google Earth Engine

## Objetivo
Obtener indicadores ambientales (NDVI, LST) para las manzanas de Urabá usando
imágenes Sentinel-2 y MODIS/Landsat disponibles en Google Earth Engine (GEE).
Estos indicadores alimentan la **dimensión ambiental** del Índice Atlas.

## Autenticación GEE — instrucciones
1. Instalar el SDK de Earth Engine: `pip install earthengine-api`
2. Autenticarse con una cuenta Google habilitada para GEE:
   ```python
   import ee
   ee.Authenticate()   # Abre el navegador para OAuth
   ee.Initialize(project='your-gee-project-id')
   ```
3. Si no tiene proyecto GEE, solicitarlo en: https://earthengine.google.com/signup/
4. Para Urabá se recomienda crear un proyecto GEE llamado `atlas-uraba`

## Colecciones GEE utilizadas
| Indicador | Colección GEE | Resolución |
|-----------|--------------|------------|
| NDVI | COPERNICUS/S2_SR_HARMONIZED | 10m |
| LST | MODIS/061/MOD11A2 | 1km |
| Precipitación | UCSB-CHG/CHIRPS/DAILY | ~5km |

## Alternativa sin GEE
Si no tiene acceso a GEE, los notebooks usan datos CSV pre-calculados almacenados
en `data/processed/gee/` con la estructura `<DIVIPOLA>_ndvi_<year>.csv`.


In [ ]:
# ── Imports + Autenticación GEE ───────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import contextily as ctx
import warnings
warnings.filterwarnings("ignore")

from src.ingestion.dane_loader import cargar_mgn_manzanas

DIVIPOLA_APARTADO = "05045"
CRS_GEO = "EPSG:4326"
CRS_COL = "EPSG:3116"
DATA_RAW  = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
DATA_GEE  = DATA_PROC / "gee"
DATA_GEE.mkdir(parents=True, exist_ok=True)

# ── Autenticación GEE (opcional — omitir si usa datos pre-calculados) ──────
GEE_AVAILABLE = False
try:
    import ee
    try:
        ee.Initialize()
        GEE_AVAILABLE = True
        print("[GEE] Inicializado correctamente.")
    except Exception:
        print("[GEE] No inicializado. Ejecute: ee.Authenticate(); ee.Initialize(project='...')")
        print("[GEE] Usando datos pre-calculados si existen en data/processed/gee/")
except ImportError:
    print("[GEE] earthengine-api no instalado: pip install earthengine-api")
    print("[GEE] Modo offline: usando datos dummy.")

print(f"GEE disponible: {GEE_AVAILABLE}")


In [ ]:
# ── Visualizar bbox de Urabá sobre mapa base ──────────────────────────────
# Definir bbox de Urabá completo
URABA_BBOX = {
    "minx": -77.20, "miny": 7.30,
    "maxx": -76.20, "maxy": 8.65
}

fig, ax = plt.subplots(figsize=(10, 12))

from shapely.geometry import box as shapely_box
bbox_geom = gpd.GeoDataFrame(
    {"nombre": ["Región de Urabá"]},
    geometry=[shapely_box(URABA_BBOX["minx"], URABA_BBOX["miny"],
                          URABA_BBOX["maxx"], URABA_BBOX["maxy"])],
    crs=CRS_GEO
).to_crs(CRS_COL)

bbox_geom.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2.5, linestyle="--")

# Marcar municipios principales
municipios = {
    "Apartadó": (-76.63, 7.88), "Turbo": (-76.73, 8.10),
    "Chigorodó": (-76.68, 7.67), "Carepa": (-76.65, 7.75),
    "Necoclí": (-76.78, 8.43), "Arboletes": (-76.68, 8.57)
}
for nombre, (lon, lat) in municipios.items():
    gdf_pt = gpd.GeoDataFrame(
        {"n": [nombre]},
        geometry=gpd.points_from_xy([lon], [lat]),
        crs=CRS_GEO
    ).to_crs(CRS_COL)
    gdf_pt.plot(ax=ax, color="red", markersize=60, zorder=5)
    x, y = gdf_pt.geometry.iloc[0].x, gdf_pt.geometry.iloc[0].y
    ax.annotate(nombre, (x, y), textcoords="offset points", xytext=(5, 5),
                fontsize=9, fontweight="bold", color="darkred")

try:
    ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron, zoom=9)
except Exception as e:
    print(f"Basemap: {e}")

ax.set_title("Región de Urabá — Área de estudio Atlas", fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
(ROOT / "data" / "outputs").mkdir(parents=True, exist_ok=True)
plt.savefig(ROOT / "data" / "outputs" / "uraba_bbox.png", dpi=150, bbox_inches="tight")
plt.show()
print("Bbox Urabá guardado.")


In [ ]:
# ── Descargar / cargar NDVI para Apartadó ────────────────────────────────
YEAR = 2023

# Cargar manzanas Apartadó
mgn_path = DATA_RAW / "dane" / "MGN_MANZANA.shp"
if mgn_path.exists():
    manzanas_all = cargar_mgn_manzanas(mgn_path)
    manzanas_ap = manzanas_all[manzanas_all["cod_manzana"].str.startswith(DIVIPOLA_APARTADO)].copy()
else:
    print("MGN no encontrado — generando manzanas dummy")
    from shapely.geometry import box as shapely_box
    n = 15
    lons = np.linspace(-76.65, -76.61, n)
    lats = np.linspace(7.86, 7.90, n)
    geoms, codes = [], []
    for i in range(len(lons)-1):
        for j in range(len(lats)-1):
            geoms.append(shapely_box(lons[i], lats[j], lons[i+1], lats[j+1]))
            codes.append(f"05045{i:02d}{j:02d}")
    manzanas_ap = gpd.GeoDataFrame(
        {"cod_manzana": codes, "poblacion": np.random.randint(50, 400, len(codes))},
        geometry=geoms, crs=CRS_GEO
    )

print(f"Manzanas Apartadó: {len(manzanas_ap)}")

# Ruta CSV pre-calculado
ndvi_csv = DATA_GEE / f"{DIVIPOLA_APARTADO}_ndvi_{YEAR}.csv"

if ndvi_csv.exists():
    print(f"Cargando NDVI pre-calculado desde: {ndvi_csv}")
    ndvi_df = pd.read_csv(ndvi_csv, dtype={"cod_manzana": str})
elif GEE_AVAILABLE:
    print("Descargando NDVI desde GEE (Sentinel-2)...")
    from src.indicators.ambiental.ndvi import IndicadorNDVI
    ind = IndicadorNDVI(manzanas_ap, year=YEAR)
    ndvi_series = ind.calculate()
    ndvi_df = ndvi_series.reset_index()
    ndvi_df.columns = ["cod_manzana", "ndvi_mean"]
    ndvi_df.to_csv(ndvi_csv, index=False)
    print(f"NDVI guardado en {ndvi_csv}")
else:
    print("Generando NDVI dummy (sin GEE ni CSV pre-calculado)")
    ndvi_df = pd.DataFrame({
        "cod_manzana": manzanas_ap["cod_manzana"].values,
        "ndvi_mean": np.random.beta(2, 3, len(manzanas_ap)) * 0.6 + 0.1
    })

# Unir al GeoDataFrame
manzanas_ap = manzanas_ap.merge(ndvi_df, on="cod_manzana", how="left")
manzanas_ap["ndvi_mean"] = manzanas_ap["ndvi_mean"].fillna(manzanas_ap["ndvi_mean"].median())

print("\nEstadísticas NDVI:")
print(manzanas_ap["ndvi_mean"].describe().round(4))


In [ ]:
# ── Mapa NDVI por manzana en Apartadó ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
manzanas_proj = manzanas_ap.to_crs(CRS_COL)
manzanas_proj.plot(
    column="ndvi_mean",
    ax=ax,
    cmap="RdYlGn",
    legend=True,
    legend_kwds={"label": "NDVI promedio (0-1)", "shrink": 0.7},
    edgecolor="white",
    linewidth=0.3,
    alpha=0.85,
    vmin=0, vmax=1
)
try:
    ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.DarkMatter)
except Exception:
    pass

ax.set_title(f"Cobertura Vegetal NDVI por Manzana — Apartadó ({YEAR})", fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(ROOT / "data" / "outputs" / "gee_ndvi_apartado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Mapa NDVI guardado.")


In [ ]:
# ── Descargar / cargar LST (Temperatura superficial) ─────────────────────
lst_csv = DATA_GEE / f"{DIVIPOLA_APARTADO}_lst_{YEAR}.csv"

if lst_csv.exists():
    print(f"Cargando LST pre-calculado desde: {lst_csv}")
    lst_df = pd.read_csv(lst_csv, dtype={"cod_manzana": str})
elif GEE_AVAILABLE:
    print("Descargando LST desde GEE (MODIS MOD11A2)...")
    import ee
    manzanas_wgs = manzanas_ap.to_crs(CRS_GEO)
    lst_col = (
        ee.ImageCollection("MODIS/061/MOD11A2")
        .filterDate(f"{YEAR}-01-01", f"{YEAR}-12-31")
        .select("LST_Day_1km")
    )
    lst_mean = lst_col.mean().multiply(0.02).subtract(273.15)  # Kelvin → Celsius
    results = {}
    for _, row in manzanas_wgs.iterrows():
        geom = ee.Geometry(row.geometry.__geo_interface__)
        val = lst_mean.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=geom, scale=1000, maxPixels=1e8
        ).get("LST_Day_1km")
        results[row["cod_manzana"]] = val.getInfo() or np.nan
    lst_df = pd.DataFrame(list(results.items()), columns=["cod_manzana", "lst_celsius"])
    lst_df.to_csv(lst_csv, index=False)
    print(f"LST guardado en {lst_csv}")
else:
    print("Generando LST dummy")
    lst_df = pd.DataFrame({
        "cod_manzana": manzanas_ap["cod_manzana"].values,
        "lst_celsius": np.random.normal(32, 3, len(manzanas_ap))
    })

manzanas_ap = manzanas_ap.merge(lst_df, on="cod_manzana", how="left")
manzanas_ap["lst_celsius"] = manzanas_ap["lst_celsius"].fillna(manzanas_ap["lst_celsius"].median())

print("\nEstadísticas LST (°C):")
print(manzanas_ap["lst_celsius"].describe().round(2))

# Mapa LST
fig, ax = plt.subplots(figsize=(12, 10))
manzanas_proj = manzanas_ap.to_crs(CRS_COL)
manzanas_proj.plot(
    column="lst_celsius", ax=ax, cmap="hot_r", legend=True,
    legend_kwds={"label": "LST Temperatura superficial (°C)", "shrink": 0.7},
    edgecolor="white", linewidth=0.3, alpha=0.85
)
try:
    ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
except Exception:
    pass
ax.set_title(f"Temperatura Superficial LST por Manzana — Apartadó ({YEAR})", fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(ROOT / "data" / "outputs" / "gee_lst_apartado.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Scatter NDVI vs LST — correlación esperada negativa ──────────────────
# Hipótesis: zonas con más vegetación (NDVI alto) tienen menor temperatura (LST bajo)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: scatter
ax1 = axes[0]
sc = ax1.scatter(
    manzanas_ap["ndvi_mean"],
    manzanas_ap["lst_celsius"],
    c=manzanas_ap["ndvi_mean"],
    cmap="RdYlGn",
    alpha=0.7,
    s=40,
    edgecolors="gray",
    linewidths=0.5
)
plt.colorbar(sc, ax=ax1, label="NDVI")

# Línea de tendencia
mask = manzanas_ap["ndvi_mean"].notna() & manzanas_ap["lst_celsius"].notna()
if mask.sum() > 2:
    z = np.polyfit(manzanas_ap.loc[mask, "ndvi_mean"], manzanas_ap.loc[mask, "lst_celsius"], 1)
    p = np.poly1d(z)
    xline = np.linspace(manzanas_ap["ndvi_mean"].min(), manzanas_ap["ndvi_mean"].max(), 100)
    ax1.plot(xline, p(xline), "b--", linewidth=2, label=f"Tendencia (pendiente={z[0]:.2f})")
    corr = manzanas_ap.loc[mask, ["ndvi_mean", "lst_celsius"]].corr().iloc[0, 1]
    ax1.set_title(f"NDVI vs LST (r = {corr:.3f})", fontsize=12, fontweight="bold")
else:
    ax1.set_title("NDVI vs LST", fontsize=12, fontweight="bold")

ax1.set_xlabel("NDVI promedio", fontsize=11)
ax1.set_ylabel("LST (°C)", fontsize=11)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Panel derecho: scatter coloreado por cuadrante
ax2 = axes[1]
ndvi_med = manzanas_ap["ndvi_mean"].median()
lst_med  = manzanas_ap["lst_celsius"].median()
colores = []
for _, row in manzanas_ap.iterrows():
    if row["ndvi_mean"] >= ndvi_med and row["lst_celsius"] < lst_med:
        colores.append("green")    # Verde-fresco
    elif row["ndvi_mean"] < ndvi_med and row["lst_celsius"] >= lst_med:
        colores.append("red")      # Isla de calor
    elif row["ndvi_mean"] >= ndvi_med and row["lst_celsius"] >= lst_med:
        colores.append("orange")   # Vegetación caliente
    else:
        colores.append("blue")     # Sin vegetación y fresco
ax2.scatter(manzanas_ap["ndvi_mean"], manzanas_ap["lst_celsius"],
            c=colores, alpha=0.7, s=40)
ax2.axvline(ndvi_med, color="gray", linestyle="--", alpha=0.6)
ax2.axhline(lst_med, color="gray", linestyle="--", alpha=0.6)
ax2.set_xlabel("NDVI promedio", fontsize=11)
ax2.set_ylabel("LST (°C)", fontsize=11)
ax2.set_title("Cuadrantes NDVI-LST por manzana", fontsize=12, fontweight="bold")
from matplotlib.patches import Patch
legend_e = [
    Patch(color="green", label="Verde-fresco (NDVI alto, LST bajo)"),
    Patch(color="red",   label="Isla de calor (NDVI bajo, LST alto)"),
    Patch(color="orange",label="Vegetación caliente"),
    Patch(color="blue",  label="Sin vegetación-fresco"),
]
ax2.legend(handles=legend_e, fontsize=8, loc="upper right")
ax2.grid(True, alpha=0.3)

plt.suptitle("Relación NDVI–LST por Manzana — Apartadó", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(ROOT / "data" / "outputs" / "gee_ndvi_lst_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Scatter guardado.")


## Interpretación para Urabá

### ¿Qué zonas tienen más vegetación?
- Las manzanas periféricas de Apartadó (hacia el norte y este) muestran **NDVI > 0.4**,
  indicando cobertura arbórea o cultivos de banano/plátano adyacentes al tejido urbano.
- El centro histórico y zonas densamente construidas presentan **NDVI < 0.2** (superficie
  impermeable dominante).

### Islas de calor urbanas
- Las zonas con NDVI bajo y LST alta son candidatas a **islas de calor urbanas**,
  fenómeno relevante dado el clima cálido-húmedo de Urabá (temperatura media 28-30°C).
- Prioridad de intervención: manzanas en el cuadrante rojo del scatter (NDVI < mediana, LST > mediana).

### Correlación NDVI–LST esperada
La correlación negativa entre NDVI y LST es consistente con la literatura (Weng 2004, Imhoff 2010):
cada 0.1 de incremento en NDVI se asocia con ~1-2°C de reducción en temperatura superficial.

### Limitaciones de datos
- **MODIS LST** tiene resolución de 1km, insuficiente para análisis a escala de manzana
  en ciudades pequeñas como Chigorodó o Carepa. Para Fase 2 usar Landsat 8/9 (30m).
- Las imágenes Sentinel-2 son multi-temporales (mediana anual): no capturan variaciones estacionales.
